# Applying DKM palettization to an MNIST model

This tutorial shows how to palettize an MNIST model with **Differentiable K-Means (DKM)**, which learns the palettization codebook and cluster assignments during training rather than fixing them from a single one-shot K-means fit.

By the end of this tutorial you should be familiar with:
1. [How to configure DKM palettization](#DKM-Palettization)
2. [How to fine-tune a DKM-palettized model](#Fine-tuning-with-DKM)
3. [How to export the palettized model to Core AI](#Export-to-Core-AI)

**Table of Contents:**

- [Setup](#Setup)
  - [MNIST Dataset download](#MNIST-Dataset-download)
  - [Model definition](#Model-definition)
  - [Training and Evaluation](#Training-and-Evaluation)
  - [Train baseline model](#Train-baseline-model)
- [DKM Palettization](#DKM-Palettization)
  - [Fine-tuning with DKM](#Fine-tuning-with-DKM)
- [Export to Core AI](#Export-to-Core-AI)

## Setup

We train a small CNN on MNIST to get a baseline, then palettize it to 4-bit with DKM and fine-tune so the codebooks are learned jointly with the model.

In [2]:
import logging
import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchinfo import summary
from torchvision import datasets, transforms
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)

In [3]:
# Use CUDA when available; otherwise fall back to CPU.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Using device {DEVICE}")

Using device cpu


In [4]:
SEED = 1976

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [5]:
# Used to save intermediate results and datasets
SAVE_DIRECTORY = "."

### MNIST Dataset download

Helper to download the MNIST dataset with standard normalization applied.

In [6]:
def mnist_transforms() -> transforms.Compose:
    return transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])


def download_mnist_dataset(
    download_path: Path, transform: transforms.Compose | None = None
) -> tuple[datasets.MNIST, datasets.MNIST]:
    if transform is None:
        transform = mnist_transforms()
    train = datasets.MNIST(download_path, train=True, download=True, transform=transform)
    test = datasets.MNIST(download_path, train=False, download=True, transform=transform)
    return train, test

### Model definition

A simple CNN with a single Conv2d → ReLU → MaxPool block, followed by Flatten and a Linear classifier.

In [7]:
class MnistNetwork(nn.Module):
    def __init__(self, num_classes: int = 10, state_dict: dict | None = None) -> None:
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 12, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2, padding=0),
            nn.Flatten(),
            nn.Linear(2352, num_classes),
        )
        if state_dict is not None:
            self.load_state_dict(state_dict)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

### Training and Evaluation

Standard PyTorch training loop and evaluation function that computes accuracy.

In [8]:
def train_step(model, optimizer, loss_fn, inputs, ground_truth) -> float:
    model.train()
    device = next(model.parameters()).device
    inputs = inputs.to(device)
    ground_truth = ground_truth.to(device)
    optimizer.zero_grad()
    predictions = model(inputs)
    loss = loss_fn(predictions, ground_truth)
    loss.backward()
    optimizer.step()
    return loss.item()


def train_epoch(model, train_loader, optimizer, loss_fn) -> float:
    total_loss = 0.0
    for inputs, ground_truth in train_loader:
        loss = train_step(model, optimizer, loss_fn, inputs, ground_truth)
        total_loss += loss
    return total_loss / len(train_loader)


def create_adam_optimizer(model: nn.Module, lr: float = 1e-3) -> torch.optim.Adam:
    return torch.optim.Adam(model.parameters(), lr=lr)


def eval_model(model: nn.Module, test_dataloader: DataLoader) -> float:
    model.eval()
    device = next(model.parameters()).device
    num_correct = 0
    total = 0
    with torch.no_grad():
        for inputs, ground_truth in test_dataloader:
            inputs = inputs.to(device)
            ground_truth = ground_truth.to(device)
            predictions = model(inputs)
            _, predicted = torch.max(predictions.data, 1)
            total += ground_truth.size(0)
            num_correct += (predicted == ground_truth).sum().item()
    return num_correct / total

In [9]:
# Download and instantiate datasets
DOWNLOAD_PATH = Path(SAVE_DIRECTORY) / ".mnist_dataset"

train_dataset, test_dataset = download_mnist_dataset(download_path=DOWNLOAD_PATH)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:00<00:00, 13.9MB/s]
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 441kB/s]
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:00<00:00, 4.67MB/s]
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 1.83MB/s]


In [10]:
BATCH_SIZE = 128

# Instantiate dataloaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE)

The CNN model used for this tutorial contains a single Conv2d, ReLU, MaxPool, Flatten, and Linear layer. Here's the structure:

In [11]:
basic_cnn_model = MnistNetwork(num_classes=10)

# Print summary of model
summary(basic_cnn_model, input_size=(1, 1, 28, 28))

Layer (type:depth-idx)                   Output Shape              Param #
MnistNetwork                             [1, 10]                   --
├─Sequential: 1-1                        [1, 10]                   --
│    └─Conv2d: 2-1                       [1, 12, 28, 28]           120
│    └─ReLU: 2-2                         [1, 12, 28, 28]           --
│    └─MaxPool2d: 2-3                    [1, 12, 14, 14]           --
│    └─Flatten: 2-4                      [1, 2352]                 --
│    └─Linear: 2-5                       [1, 10]                   23,530
Total params: 23,650
Trainable params: 23,650
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.12
Input size (MB): 0.00
Forward/backward pass size (MB): 0.08
Params size (MB): 0.09
Estimated Total Size (MB): 0.17

### Train baseline model

Let's train this model so we can get a baseline accuracy. We save the trained weights so we can reload them for the palettization experiment.

In [12]:
EPOCHS = 10

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = create_adam_optimizer(basic_cnn_model)

basic_cnn_model = basic_cnn_model.to(DEVICE)

epoch_results = []
for epoch in tqdm(range(EPOCHS)):
    epoch_avg_loss = train_epoch(
        model=basic_cnn_model, train_loader=train_loader, optimizer=optimizer, loss_fn=loss_fn
    )
    epoch_results.append(f"  Epoch {epoch + 1}: loss={epoch_avg_loss:.4f}")

basic_cnn_model = basic_cnn_model.cpu().eval()
print("\n".join(epoch_results))

# Save trained weights for reuse in the palettization section
pretrained_state_dict = basic_cnn_model.state_dict()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [01:00<00:00,  6.05s/it]

  Epoch 1: loss=0.3126
  Epoch 2: loss=0.1170
  Epoch 3: loss=0.0821
  Epoch 4: loss=0.0669
  Epoch 5: loss=0.0590
  Epoch 6: loss=0.0522
  Epoch 7: loss=0.0481
  Epoch 8: loss=0.0452
  Epoch 9: loss=0.0415
  Epoch 10: loss=0.0379


In [13]:
accuracy = eval_model(basic_cnn_model, test_loader)
print(f"Baseline accuracy: {accuracy:.4f}")

Baseline accuracy: 0.9798


## DKM Palettization

Palettization compresses a model's weights by clustering them into a small look-up table (LUT) of centroids, and replacing each weight with an index into that table.

Plain K-means palettization computes the LUT once, post-training, and freezes it. DKM instead makes the clustering step differentiable: during training it uses a soft assignment of weights to centroids, so gradients flow back into the palettized weights and their codebooks. This lets the codebook and assignments be learned jointly with the rest of the model, which can recover accuracy that a one-shot fit leaves behind.

DKM is enabled by setting the `training_strategy_spec` field of a `PalettizationSpec` to a `DKMTrainingSpec`. The same `KMeansPalettizer` drives it.

In [14]:
from coreai_opt.palettization import (
    KMeansPalettizer,
    KMeansPalettizerConfig,
    ModuleKMeansPalettizerConfig,
    PalettizationSpec,
)
from coreai_opt.palettization.kmeans.dkm import DKMTrainingSpec
from coreai_opt.palettization.spec import PerGroupedChannelGranularity

example_inputs = (torch.randn(1, 1, 28, 28),)

W1030 11:30:48.699000 8891 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W1030 11:30:48.728000 8891 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


We use 4-bit **per-grouped-channel** palettization (each group of 2 output channels gets its own LUT of 2⁴ = 16 centroids), and attach a `DKMTrainingSpec` so the palettization is trained with DKM.

`DKMTrainingSpec` exposes a few hyperparameters (all optional): `tau` (softmax temperature for the soft assignment), `kmeans_max_iter` (soft-assignment iterations per forward), `enforce_zero`, and `zero_threshold`. Here we keep the defaults.

In [15]:
dkm_model = MnistNetwork(num_classes=10, state_dict=pretrained_state_dict)
dkm_model.eval()

weight_spec = PalettizationSpec(
    n_bits=4,
    granularity=PerGroupedChannelGranularity(axis=0, group_size=2),
    training_strategy_spec=DKMTrainingSpec(),
)

dkm_config = KMeansPalettizerConfig(
    global_config=ModuleKMeansPalettizerConfig(
        op_state_spec={"weight": weight_spec},
    )
)

dkm_palettizer = KMeansPalettizer(dkm_model, dkm_config)
dkm_prepared = dkm_palettizer.prepare(example_inputs)
print("Prepared DKM palettization with n_bits=4")

# Move the prepared model onto the target device so DKM's training-time forward runs there.
# prepare() itself runs on CPU: k-means centroid initialization uses a CPU extension.
dkm_prepared = dkm_prepared.to(DEVICE)

INFO:coreai_opt.palettization.kmeans.palettizer:Preparing model for palettization
Palettizing layers (num_workers=1): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.16it/s]

Prepared DKM palettization with n_bits=4


After `prepare()`, the model has fake-palettization modules whose codebooks are initialized from one-shot K-means. Measure the accuracy of the palettized model before DKM fine-tuning:

In [16]:
dkm_initial_accuracy = eval_model(dkm_prepared, test_loader)
print(f"DKM (before fine-tuning) accuracy: {dkm_initial_accuracy:.4f}")

DKM (before fine-tuning) accuracy: 0.9789


### Fine-tuning with DKM

We now fine-tune inside `dkm_palettizer.training_mode()`. Within this context each forward pass runs DKM's differentiable soft assignment, so gradients flow into the palettized weights and their codebooks are updated by the optimizer.

`training_mode()` also restores the model's original train/eval state on exit, so we can evaluate immediately afterwards.

In [17]:
import torch.nn.functional as F

DKM_FINETUNE_STEPS = 200
optimizer = create_adam_optimizer(dkm_prepared, lr=1e-4)

losses = []
step = 0
with dkm_palettizer.training_mode():
    while step < DKM_FINETUNE_STEPS:
        for inputs, targets in tqdm(train_loader):
            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE)
            optimizer.zero_grad()
            loss = F.cross_entropy(dkm_prepared(inputs), targets)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            step += 1
            if step >= DKM_FINETUNE_STEPS:
                break

print(f"Fine-tuned DKM for {step} steps; loss {losses[0]:.4f} -> {losses[-1]:.4f}")

 42%|█████████████████████████████████████████████████████████▎                                                                             | 199/469 [00:03<00:04, 56.70it/s]

Fine-tuned DKM for 200 steps; loss 0.0903 -> 0.0175


In [18]:
dkm_accuracy = eval_model(dkm_prepared, test_loader)
print(f"DKM (after fine-tuning) accuracy: {dkm_accuracy:.4f}")

DKM (after fine-tuning) accuracy: 0.9827


## Export to Core AI

Once the palettized model is ready, call `finalize()` to convert the fake-palettization modules into the real LUT-based representation for deployment.

Pass `ExportBackend.CoreAI` to `finalize(backend=...)` to target the `.aimodel` format produced by `coreai-torch`.

In [19]:
from coreai_opt import ExportBackend

# Export (torch.export + Core AI conversion) runs on CPU and traces with the CPU
# example_inputs, so move the model back off the GPU first.
dkm_prepared = dkm_prepared.to("cpu")
coreai_model = dkm_palettizer.finalize(backend=ExportBackend.CoreAI)

INFO:coreai_opt.palettization.kmeans.palettizer:Finalizing model for backend: coreai
/Volumes/Data/coreai-opt/external/src/coreai_opt/palettization/kmeans/_prepare_for_export.py:302: UserWarning: coreai-torch has only been validated with torch<=2.13.0; found torch 2.14.0. Some functionality may not work as expected.
  from coreai_torch._compression.custom_layers import (  # noqa: PLC0415


The export proceeds in three steps:

- Trace the model with `torch.export.export()` to obtain a graph representation.
- Apply `cast_to_16_bit_precision()` to cast remaining FP32 parameters to FP16 for optimal on-device performance.
- Convert the exported program to Core AI format using `coreai-torch.TorchConverter`.

In [20]:
import shutil

from coreai_opt.casting import cast_to_16_bit_precision
from coreai_torch import TorchConverter, get_decomp_table

exported_program = torch.export.export(coreai_model, example_inputs, strict=False)
exported_program = exported_program.run_decompositions(get_decomp_table())
cast_to_16_bit_precision(exported_program)

coreai_program = TorchConverter().add_exported_program(exported_program).to_coreai()

output_path = Path(SAVE_DIRECTORY) / "exported_model.aimodel"
if output_path.exists():
    shutil.rmtree(output_path)
coreai_program.save_asset(output_path)
print(f"Exported: {output_path}")

coreai-torch 0.0.0.dev202610011705+745fc1b: converting 1 program(s) to Core AI

Exported: exported_model.aimodel
